In [ ]:
!pip install -q tensorflow
!pip install -q numpy
!pip install -q pandas
!pip install -q matplotlib
!pip install -q pillow
!pip install -q tqdm

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm

from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array

from google.colab import files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
zip_path = "/content/drive/MyDrive/DL ASS/archive (1).zip"

In [ ]:
import zipfile

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/CAP_data")

In [ ]:
import os

def load_doc(filename):
    file = open(filename, 'r')
    text = file.read()
    file.close()
    return text

caption_file = "/content/CAP_data/captions.txt" # Changed filename from Flickr8k.token.txt to captions.txt

doc = load_doc(caption_file)

print(doc[:500])


In [ ]:
def load_descriptions(doc):
    mapping = dict()

    for line in doc.split('\n'):
        tokens = line.split()

        if len(line) < 2:
            continue

        image_id = tokens[0].split('.')[0]
        image_desc = ' '.join(tokens[1:])

        if image_id not in mapping:
            mapping[image_id] = []

        mapping[image_id].append(image_desc)

    return mapping


descriptions = load_descriptions(doc)

print("Total Images:", len(descriptions))

In [ ]:
def clean_descriptions(descriptions):
    for key, desc_list in descriptions.items():
        for i in range(len(desc_list)):
            desc = desc_list[i]

            desc = desc.lower()
            desc = desc.split()

            desc = [word for word in desc if word.isalpha()]
            desc = [word for word in desc if len(word) > 1]

            desc_list[i] = "startseq " + " ".join(desc) + " endseq"


clean_descriptions(descriptions)

print(descriptions[list(descriptions.keys())[0]])

In [ ]:
from tensorflow.keras.models import Model

base_model = VGG16()

feature_model = Model(
    inputs=base_model.inputs,
    outputs=base_model.layers[-2].output
)

print(feature_model.summary())

In [ ]:
def extract_features(directory):
    features = dict()

    for name in tqdm(os.listdir(directory)):
        # Ensure we only process image files
        if not (name.endswith('.jpg') or name.endswith('.jpeg') or name.endswith('.png')):
            continue

        filename = directory + "/" + name

        image = load_img(
            filename,
            target_size=(224, 224)
        )

        image = img_to_array(image)

        image = image.reshape(
            (1, image.shape[0],
             image.shape[1],
             image.shape[2])
        )

        image = preprocess_input(image)

        feature = feature_model.predict(
            image,
            verbose=0
        )

        image_id = name.split('.')[0]
        features[image_id] = feature

    return features


# Define the path to your image directory
image_path = "/content/CAP_data/images" # Updated path to the correct subdirectory

features = extract_features(image_path)

print("Features Extracted:", len(features))

In [ ]:
import os
print(os.listdir('/content/CAP_data'))

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

def to_lines(descriptions):
    all_desc = []

    for key in descriptions.keys():
        for desc in descriptions[key]:
            all_desc.append(desc)

    return all_desc


def create_tokenizer(descriptions):
    lines = to_lines(descriptions)

    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(lines)

    return tokenizer


tokenizer = create_tokenizer(descriptions)

vocab_size = len(tokenizer.word_index) + 1

print("Vocabulary Size:", vocab_size)

In [ ]:
def max_length(descriptions):
    lines = to_lines(descriptions)

    return max(
        len(d.split())
        for d in lines
    )


max_len = max_length(descriptions)

print("Maximum Caption Length:", max_len)

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

def data_generator(
    descriptions,
    features,
    tokenizer,
    max_length,
    vocab_size
):
    while True:
        for key, desc_list in descriptions.items():

            photo = features[key][0]

            for desc in desc_list:
                seq = tokenizer.texts_to_sequences(
                    [desc]
                )[0]

                for i in range(1, len(seq)):
                    in_seq = seq[:i]
                    out_seq = seq[i]

                    in_seq = pad_sequences(
                        [in_seq],
                        maxlen=max_length
                    )[0]

                    out_seq = to_categorical(
                        [out_seq],
                        num_classes=vocab_size
                    )[0]

                    yield [[photo, in_seq], out_seq]

In [ ]:
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import SimpleRNN
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import add
from tensorflow.keras.models import Model


# IMAGE FEATURE INPUT

inputs1 = Input(shape=(4096,))

fe1 = Dropout(0.5)(inputs1)

fe2 = Dense(
    256,
    activation='relu'
)(fe1)


# TEXT SEQUENCE INPUT

inputs2 = Input(shape=(max_len,))

se1 = Embedding(
    vocab_size,
    256,
    mask_zero=True
)(inputs2)

se2 = Dropout(0.5)(se1)

# HERE WE USE RNN

se3 = SimpleRNN(256)(se2)


# MERGE BOTH INPUTS

decoder1 = add([fe2, se3])

decoder2 = Dense(
    256,
    activation='relu'
)(decoder1)

outputs = Dense(
    vocab_size,
    activation='softmax'
)(decoder2)


# FINAL MODEL

model = Model(
    inputs=[inputs1, inputs2],
    outputs=outputs
)

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam'
)

print(model.summary())

In [ ]:
def data_generator(
    descriptions,
    features,
    tokenizer,
    max_length,
    vocab_size
):
    while True:
        for key, desc_list in descriptions.items():

            # skip if image feature not found
            if key not in features:
                continue

            photo = features[key][0]

            for desc in desc_list:
                seq = tokenizer.texts_to_sequences([desc])[0]

                for i in range(1, len(seq)):
                    in_seq = seq[:i]
                    out_seq = seq[i]

                    in_seq = pad_sequences(
                        [in_seq],
                        maxlen=max_length
                    )[0]

                    out_seq = to_categorical(
                        [out_seq],
                        num_classes=vocab_size
                    )[0]

                    yield [[photo, in_seq], out_seq
]

In [ ]:
# Remove the problematic header key if it exists in descriptions
if 'image_name|caption_number|caption_text' in descriptions:
    del descriptions['image_name|caption_number|caption_text']

steps = len(descriptions) # Recalculate steps after cleaning descriptions

generator = data_generator(
    descriptions,
    features,
    tokenizer,
    max_len,
    vocab_size
)

model.fit(
    generator,
    epochs=30,
    steps_per_epoch=steps,
    verbose=1
)

In [ ]:
model.save("image_caption_rnn_model.h5")

print("RNN Model Saved Successfully")

In [ ]:
def word_for_id(integer, tokenizer):
    for word, index in tokenizer.word_index.items():
        if index == integer:
            return word

    return None

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

def generate_desc(
    model,
    tokenizer,
    photo,
    max_length
):
    in_text = "startseq"

    for i in range(max_length):

        sequence = tokenizer.texts_to_sequences(
            [in_text]
        )[0]

        sequence = pad_sequences(
            [sequence],
            maxlen=max_length
        )

        yhat = model.predict(
            [photo, sequence],
            verbose=0
        )

        yhat = np.argmax(yhat)

        word = word_for_id(
            yhat,
            tokenizer
        )

        if word is None:
            break

        in_text += " " + word

        if word == "endseq":
            break

    return in_text

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
def extract_single_feature(filename):
    image = load_img(
        filename,
        target_size=(224, 224)
    )

    image = img_to_array(image)

    image = image.reshape(
        (1, image.shape[0],
         image.shape[1],
         image.shape[2])
    )

    image = preprocess_input(image)

    feature = feature_model.predict(
        image,
        verbose=0
    )

    return feature

In [ ]:
image_name = list(uploaded.keys())[0]

photo = extract_single_feature(image_name)

caption = generate_desc(
    model,
    tokenizer,
    photo,
    max_len
)

print("Generated Caption:")
print(caption)

In [ ]:
img = load_img(image_name)

import matplotlib.pyplot as plt

plt.imshow(img)
plt.axis("off")
plt.show()